# Optional coupon relevance experiment â€” Pandas implementation

Same rules, candidates, scores, tie-breaks and outcomes as notebook 03, using
Pandas for loading and all transformations. The main USD 2 targeting policy
is unchanged. This is historical recommendation relevance, not causal uplift.

## 1. Read sources and apply the same monetary validity policy

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "src/retail_customer_360/metrics.py").is_file()
)
OUT = ROOT / "data/processed/coupon_recommendations_pandas"
OUT.mkdir(parents=True, exist_ok=True)
raw = pd.read_csv(ROOT / "data/raw/transaction_data.csv", dtype="string")
raw.columns = raw.columns.str.lower()

In [2]:
sales = raw.sales_value
numeric = pd.to_numeric(sales, errors="coerce")
quantity = pd.to_numeric(raw.quantity, errors="raise")
regular = sales.str.fullmatch(r"-?[0-9]+([.][0-9]{1,2})?").fillna(False)
residue = numeric.abs().lt(1e-9) & quantity.eq(0)
assert (regular | residue).all(), "Invalid monetary source text"
raw["sales_positive"] = numeric.gt(0) & ~residue
keys = ["household_key", "basket_id", "day", "product_id"]
raw[keys] = raw[keys].astype("int64")
raw["quantity"] = quantity.astype("int64")
observation_end = int(raw.day.max())
display(pd.DataFrame({"last_observed_day": [observation_end]}))

,last_observed_day
0,711


All transaction columns are initially read as strings, as in the SQL loader.
The same cents-format / negligible-zero-quantity-residue rule is checked.
No revenue amounts are aggregated here: numeric conversion is used only for
validation and the positive-sales filter. No positive-quantity residual is allowed.

In [3]:
def read_source(filename):
    frame = pd.read_csv(ROOT / "data/raw" / filename)
    frame.columns = frame.columns.str.lower()
    return frame

campaigns = read_source("campaign_desc.csv")
contacts = read_source("campaign_table.csv")
mapping_raw = read_source("coupon.csv")
redemptions = read_source("coupon_redempt.csv")

## 2. Define a historical decision point

Each decision is made immediately before campaign START_DAY. Features use only
positive-quantity, positive-sales purchases in the preceding 180 days.
Require START_DAY > 180 and the entire campaign to end within observation.
Evaluate from START_DAY through min(END_DAY, START_DAY + 29), inclusive.
This can be shorter than 30 days; duration is reported per campaign.
Start/end inclusivity is an explicit working convention.

Only recorded recipients of TypeB/TypeC are eligible: the source describes their
full campaign coupon set. Exclude TypeA because individual allocations are unknown.
No claim is made that a recipient saw an offer or that its full terms are known.
The catalogue is treated as available at campaign start; creation timestamps are absent.

In [4]:
decisions = campaigns.loc[
    campaigns.description.isin(["TypeB", "TypeC"])
    & campaigns.start_day.gt(180)
    & campaigns.end_day.le(observation_end)
].copy()
decisions["outcome_end"] = np.minimum(
    decisions.end_day, decisions.start_day + 29
)
mapping = mapping_raw.drop_duplicates()
assert not decisions.empty
display(decisions.sort_values("start_day"))

,description,campaign,start_day,end_day,outcome_end
28,TypeC,27,237,300,266
27,TypeB,28,259,320,288
26,TypeB,29,281,334,310
23,TypeB,1,346,383,375
24,TypeB,2,351,383,380
20,TypeC,3,356,412,385
22,TypeB,4,372,404,401
21,TypeB,5,377,411,406
19,TypeC,6,393,425,422
18,TypeB,7,398,432,427


In [5]:
catalogue_audit = pd.DataFrame({
    "raw_mapping_rows": [len(mapping_raw)],
    "unique_mapping_rows": [len(mapping)],
    "evaluated_campaigns": [len(decisions)],
})
display(catalogue_audit)
paid = raw.loc[raw.quantity.gt(0) & raw.sales_positive, keys].drop_duplicates()
del raw, sales, numeric, quantity, regular, residue

,raw_mapping_rows,unique_mapping_rows,evaluated_campaigns
0,124548,119384,24


## 3. Generate candidates and explainable scores

One candidate is a household/campaign/coupon triple. Keep recipients with at
least one paid purchase in the prior 180 days; report this restricted population.
Every mapped campaign coupon is a candidate, including zero-affinity offers.

Personal score: number of distinct past baskets containing an eligible product.
Break ties by most recent eligible-product purchase, then historical population
popularity, then coupon identifier. Baseline: population basket count alone.
Population counts use all households, with the same strictly historical window.
Count baskets once per coupon even when several eligible products were bought.
This simple ranking favours familiar products and broad offers; it does not
identify households persuaded by a discount or optimise profit.

Process one campaign at a time to limit intermediate joins. The following small
functions separate population selection, scoring, ranking and future labelling.
Each function is called for every historical campaign; no future label affects scores.

In [6]:
def campaign_population(d):
    history = paid.loc[
        paid.day.ge(d.start_day - 180) & paid.day.lt(d.start_day)
    ]
    assert history.day.lt(d.start_day).all()
    assert history.day.ge(d.start_day - 180).all()
    recipients = contacts.loc[
        contacts.campaign.eq(d.campaign), ["household_key"]
    ].drop_duplicates()
    eligible = recipients.loc[
        recipients.household_key.isin(history.household_key)
    ]
    return history, recipients, eligible

In [7]:
def historical_scores(history, offers):
    matched = history.merge(offers, on="product_id", how="inner")
    affinity = matched.groupby(["household_key", "coupon_upc"]).agg(
        personal_baskets=("basket_id", "nunique"),
        last_product_day=("day", "max"),
    ).reset_index()
    basket_keys = ["coupon_upc", "household_key", "basket_id"]
    popularity = (
        matched[basket_keys].drop_duplicates().groupby("coupon_upc")
        .size().rename("popular_baskets").reset_index()
    )
    return affinity, popularity

In [8]:
def candidate_scores(eligible, offers, affinity, popularity, d):
    candidates = eligible.merge(
        offers[["coupon_upc"]].drop_duplicates(), how="cross"
    )
    candidates = candidates.merge(
        affinity, on=["household_key", "coupon_upc"], how="left",
        validate="one_to_one",
    ).merge(popularity, on="coupon_upc", how="left", validate="many_to_one")
    fills = {"personal_baskets": 0, "last_product_day": -1, "popular_baskets": 0}
    candidates = candidates.fillna(fills)
    candidates[list(fills)] = candidates[list(fills)].astype("int64")
    for field in ["campaign", "description", "start_day", "end_day", "outcome_end"]:
        candidates[field] = getattr(d, field)
    return candidates

In [9]:
def rank_candidates(candidates):
    personal_order = [
        "household_key", "personal_baskets", "last_product_day",
        "popular_baskets", "coupon_upc",
    ]
    ranked = candidates.sort_values(
        personal_order, ascending=[True, False, False, False, True]
    ).copy()
    ranked["personal_rank"] = ranked.groupby("household_key").cumcount() + 1
    baseline = ranked.sort_values(
        ["household_key", "popular_baskets", "coupon_upc"],
        ascending=[True, False, True],
    ).copy()
    baseline["baseline_rank"] = baseline.groupby("household_key").cumcount() + 1
    ranked["baseline_rank"] = baseline.baseline_rank.reindex(ranked.index)
    return ranked

## 4. Reveal future outcomes only after ranking

A positive relevance label means at least one eligible product was purchased
in a paid basket during the evaluation window. A redemption label is reported
separately and uses the same window; it is not inferred from product purchases.
Both labels use EXISTS, avoiding inflation from many-to-many coupon mappings.
Overlapping coupons can receive a positive label from the same purchase.

In [10]:
def reveal_outcomes(ranked, offers, d):
    future = paid.loc[paid.day.between(d.start_day, d.outcome_end)]
    hits = future.merge(offers, on="product_id", how="inner")
    pair = ["household_key", "coupon_upc"]
    hits = hits[pair].drop_duplicates().assign(product_hit=1)
    redeemed = redemptions.loc[
        redemptions.campaign.eq(d.campaign)
        & redemptions.day.between(d.start_day, d.outcome_end), pair
    ].drop_duplicates().assign(redeemed=1)
    evaluated = ranked.merge(hits, on=pair, how="left", validate="one_to_one")
    evaluated = evaluated.merge(
        redeemed, on=pair, how="left", validate="one_to_one"
    )
    labels = ["product_hit", "redeemed"]
    evaluated[labels] = evaluated[labels].fillna(0).astype("int64")
    return evaluated

In [11]:
evaluations, coverage_rows = [], []
for d in decisions.itertuples(index=False):
    history, recipients, eligible = campaign_population(d)
    offers = mapping.loc[mapping.campaign.eq(d.campaign)].drop(columns="campaign")
    affinity, popularity = historical_scores(history, offers)
    candidates = candidate_scores(eligible, offers, affinity, popularity, d)
    ranked_part = rank_candidates(candidates)
    evaluations.append(reveal_outcomes(ranked_part, offers, d))
    coverage_rows.append({
        "campaign": d.campaign, "start_day": d.start_day,
        "days": d.outcome_end - d.start_day + 1,
        "recorded_recipients": len(recipients),
        "eligible_recipients": len(eligible),
    })
evaluated = pd.concat(evaluations, ignore_index=True)
coverage = pd.DataFrame(coverage_rows).sort_values("start_day")
display(coverage)

,campaign,start_day,days,recorded_recipients,eligible_recipients
23,27,237,30,12,12
22,28,259,30,17,17
21,29,281,30,118,118
19,1,346,30,13,13
20,2,351,30,48,48
16,3,356,30,12,12
18,4,372,30,81,81
17,5,377,30,166,166
15,6,393,30,65,65
14,7,398,30,198,198


## 5. Compare top-three recommendations with popularity

Precision@3 is the fraction of selected offers with a positive product label.
Hit rate@3 is the fraction of household/campaign decisions with at least one hit.
Use min(3, available offers) if fewer than three candidates exist.
Average decisions equally; households appearing in several campaigns repeat.
Redemption precision is a descriptive secondary metric, not a response probability.
Rules and K=3 are fixed in this notebook, not tuned against these outcomes.

In [12]:
selected = pd.concat([
    evaluated.loc[evaluated.personal_rank.le(3)].assign(method="personal_history"),
    evaluated.loc[evaluated.baseline_rank.le(3)].assign(
        method="population_popularity"
    ),
], ignore_index=True)
per_decision = selected.groupby(["method", "campaign", "household_key"]).agg(
    recommended=("coupon_upc", "size"),
    precision_at_3=("product_hit", "mean"),
    hit_at_3=("product_hit", "max"),
    redemption_precision_at_3=("redeemed", "mean"),
).reset_index()

In [13]:
metrics = ["precision_at_3", "hit_at_3", "redemption_precision_at_3"]
summary = per_decision.groupby("method")[metrics].mean()
summary["decisions"] = per_decision.groupby("method").size()
by_campaign = per_decision.groupby(["campaign", "method"])[metrics].mean()
display(summary.round(4))
display(by_campaign.round(4))

,precision_at_3,hit_at_3,redemption_precision_at_3,decisions
method,,,,
personal_history,0.1918,0.4052,0.0203,3129
population_popularity,0.1364,0.3247,0.0154,3129


precision_at_3  hit_at_3  \
campaign method                                            
1        personal_history               0.1282    0.3846   
         population_popularity          0.1282    0.3077   
2        personal_history               0.1458    0.3542   
         population_popularity          0.0972    0.2708   
3        personal_history               0.1111    0.2500   
         population_popularity          0.0833    0.1667   
4        personal_history               0.1564    0.3951   
         population_popularity          0.0658    0.1975   
5        personal_history               0.0904    0.2048   
         population_popularity          0.0803    0.1867   
6        personal_history               0.0308    0.0308   
         population_popularity          0.0308    0.0308   
7        personal_history               0.0471    0.1313   
         population_popularity          0.0404    0.1212   
9        personal_history               0.2367    0.5000   
         population_popularity          0.1667    0.3750   
10       personal_history               0.1057    0.2439   
         population_popularity          0.0786    0.2114   
11       personal_history               0.0748    0.1869   
         population_popularity          0.0639    0.1729   
12       personal_history               0.3549    0.6941   
         population_popularity          0.3176    0.6824   
14       personal_history               0.2158    0.4598   
         population_popularity          0.1562    0.4018   
15       personal_history               0.2059    0.2941   
         population_popularity          0.2059    0.2941   
16       personal_history               0.1135    0.2660   
         population_popularity          0.0887    0.2287   
17       personal_history               0.2112    0.4752   
         population_popularity          0.1007    0.2673   
19       personal_history               0.2205    0.4615   
         population_popularity          0.1718    0.3846   
20       personal_history               0.1844    0.3689   
         population_popularity          0.1311    0.3197   
21       personal_history               0.1077    0.3077   
         population_popularity          0.0564    0.1385   
22       personal_history               0.2645    0.5290   
         population_popularity          0.1775    0.4022   
23       personal_history               0.2969    0.5847   
         population_popularity          0.1876    0.4372   
25       personal_history               0.3476    0.6791   
         population_popularity          0.2478    0.5722   
27       personal_history               0.0833    0.2500   
         population_popularity          0.0556    0.1667   
28       personal_history               0.2549    0.5882   
         population_popularity          0.2353    0.5882   
29       personal_history               0.2203    0.4746   
         population_popularity          0.1384    0.3390   

                                redemption_precision_at_3  
campaign method                                            
1        personal_history                          0.0256  
         population_popularity                     0.0000  
2        personal_history                          0.0278  
         population_popularity                     0.0208  
3        personal_history                          0.0278  
         population_popularity                     0.0278  
4        personal_history                          0.0123  
         population_popularity                     0.0082  
5        personal_history                          0.0120  
         population_popularity                     0.0100  
6        personal_history                          0.0154  
         population_popularity                     0.0154  
7        personal_history                          0.0067  
         population_popularity                     0.0051  
9        personal_history                          0.0246  
         po

## 6. Checks and exports â€” same recommendation columns as SQL

In [14]:
pair = ["household_key", "campaign", "coupon_upc"]
assert not evaluated.duplicated(pair).any()
assert evaluated.outcome_end.le(observation_end).all()
assert per_decision.recommended.between(1, 3).all()
columns = [
    "household_key", "campaign", "coupon_upc", "start_day", "outcome_end",
    "personal_rank", "personal_baskets", "last_product_day", "popular_baskets",
]
recommendations = evaluated.loc[evaluated.personal_rank.le(3), columns]
recommendations = recommendations.sort_values(
    ["campaign", "household_key", "personal_rank"]
)

In [15]:
recommendations.to_csv(OUT / "recommendations.csv", index=False)
summary.to_csv(OUT / "evaluation_summary.csv")
by_campaign.to_csv(OUT / "evaluation_by_campaign.csv")
coverage.to_csv(OUT / "population_coverage.csv", index=False)
display(pd.DataFrame({"exported_recommendations": [len(recommendations)]}))

,exported_recommendations
0,9240


## 7. Verify agreement with the executed SQL notebook

Run notebook 03 first to create its reference exports. Ignore column-name casing
and numeric storage dtypes; require matching values after sorting by stable keys.
Integer values match exactly; floating aggregates use tolerance 1e-12.
This verification is excluded from benchmark timings. A separate benchmark
also compares every candidate score, both ranks and both outcome labels.

In [16]:
# Verification: excluded from benchmark timing.
reference = ROOT / "data/processed/coupon_recommendations"
files = {
    "recommendations.csv": ["campaign", "household_key", "coupon_upc"],
    "evaluation_summary.csv": ["method"],
    "evaluation_by_campaign.csv": ["campaign", "method"],
    "population_coverage.csv": ["campaign"],
}
for filename, sort_keys in files.items():
    left = pd.read_csv(reference / filename)
    right = pd.read_csv(OUT / filename)
    left.columns = left.columns.str.lower()
    right.columns = right.columns.str.lower()
    left = left.sort_values(sort_keys).reset_index(drop=True)
    right = right[left.columns].sort_values(sort_keys).reset_index(drop=True)
    pd.testing.assert_frame_equal(
        left, right, check_dtype=False, rtol=1e-12, atol=1e-12
    )
print("All four exports match (floating tolerance 1e-12).")

All four exports match (floating tolerance 1e-12).


## 7. What we can and cannot conclude

- A higher product hit rate supports relevance on these historical recipients;
  it does not show incremental sales, prevented churn or savings in marketing cost.
  Read the per-campaign results before generalising an aggregate difference.
- There is no untreated counterfactual. Purchases happened under historical
  marketing, including all received offers, not only our proposed top three.
- TypeA is excluded; inactive recipients, new households and campaigns without
  180 days of prior observation are outside this evaluation population.
- Campaign starts are separate historical cutoffs. Later campaigns may use
  earlier observed purchases legitimately; repeated households and overlapping
  windows mean the decisions are not independent statistical observations.
- Coupon face value, minimum spend, detailed conditions, funding and product
  margin are absent. Dates and product mappings are necessary, not sufficient,
  checks for operational eligibility. These coupons are historical, not live.
- Exact-product history has cold-start limits. Broad coupons have more chances
  to match a purchase. Several coupons may cover the same products; no diversity,
  redemption stacking or stock constraint is modelled.
- There is no claimed online A/B experiment, causal uplift estimate or profit
  optimisation. Main-project USD 2 scenario assumptions do not become the
  actual value of these coupon identifiers.

### Next step: a prospective test, not performed here

Validate offer terms and costs first. Randomise eligible households into no-offer,
popular-offer and personalised-offer groups, with comparable offer economics.
Keep assignment at household level and prevent overlapping experimental contacts.
Predefine the observation window, sample-size calculation, primary incremental
contribution metric and guardrails. Compare intention-to-treat outcomes, including
discount/contact costs and purchases without redemption. Until this is available,
use the notebook as a recommendation prototype rather than a spending policy.

## 8. Measured SQL / Pandas comparison

| Engine | Load/validate CSV, s | Analysis/export, s | Total, s | Peak RAM, MiB |
|---|---:|---:|---:|---:|
| pandas | 6.54 | 3.54 | 9.17 | 675 |
| sql | 1.44 | 2.29 | 3.71 | 1063 |

Medians of three fresh-process runs per engine. Pandas/SQL total-time ratio: **2.47x** on this machine.

See [benchmark methodology and results](../reports/coupon_pandas_benchmark.md) for timing boundaries, memory definition, caveats and the reproducible command.